# Introducción al Prompt Engineering con Python
## Matemática para Economistas III — UNGS
### Lic. Mateo Suster

En esta clase práctica aprendemos a diseñar **buenos prompts** para obtener mejores resultados de los modelos de lenguaje (LLMs). Usamos **Hugging Face** con Python en Google Colab.

**Agenda:**
1. ⚙️ Configuración del entorno
2. 📝 Los 4 elementos de un prompt
3. 🎛️ Parámetros del modelo (temperatura, max_tokens)
4. 🎭 Roles: system, user, assistant
5. 🎯 Claridad y especificidad
6. 🏷️ Estructura con tags XML
7. 📸 Zero-Shot, One-Shot y Few-Shot
8. 🧠 Chain-of-Thought (CoT)
9. 📋 Prompt templates con variables
10. 🔄 Síntesis: el proceso iterativo

---
## ⚙️ 0. Configuración del Entorno

In [ ]:
# Instalamos la librería de Hugging Face
!pip install -q huggingface_hub
print("Instalacion completa")

### Obtener tu token de Hugging Face

Para usar modelos de Hugging Face necesitás un **token gratuito** (sin tarjeta de crédito):

1. Creá una cuenta gratis en **huggingface.co** (solo requiere email)
2. Andá a **huggingface.co/settings/tokens**
3. Hacé click en **"New token"** → tipo **"Read"** es suficiente
4. Copiá el token y pegalo en la celda siguiente

⚠️ No compartás tu token con nadie ni lo subas a repositorios públicos.

In [ ]:
from huggingface_hub import InferenceClient

# Pegá tu token entre las comillas
HF_TOKEN = "TU_TOKEN_AQUI"

client = InferenceClient(api_key=HF_TOKEN)
print("Cliente configurado correctamente")

In [ ]:
def llamar_modelo(prompt, system=None, temperatura=0.7, max_tokens=1024):
    """
    Llama al modelo de Hugging Face con el prompt dado.

    Parámetros:
        prompt      : texto del prompt (str)
        system      : instrucción de sistema/rol del modelo (str, opcional)
        temperatura : creatividad del modelo, entre 0 (predecible) y 1 (creativo)
        max_tokens  : longitud máxima de la respuesta en tokens
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    respuesta = client.chat.completions.create(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=messages,
        temperature=temperatura,
        max_tokens=max_tokens,
    )
    return respuesta.choices[0].message.content

print("Funcion lista para usar")

In [ ]:
# Primera prueba
respuesta = llamar_modelo("Que es la inflacion? Explicalo en una sola oracion simple.")
print(respuesta)

---
## 📝 1. Anatomía de un Prompt: Los 4 Elementos

Un buen prompt generalmente incluye **4 componentes**:

| Elemento | Descripción | Ejemplo |
|---|---|---|
| **Instrucción** | Qué querés que haga el modelo | *"Analizá el siguiente texto"* |
| **Contexto** | Información de fondo para el modelo | *"La audiencia son estudiantes universitarios..."* |
| **Datos de entrada** | El contenido a procesar | *"El BCRA subió la tasa al 40%..."* |
| **Indicador de salida** | Formato o tipo de respuesta esperada | *"Respondé en 3 puntos concisos"* |

No siempre necesitás los 4, pero cuantos más incluyas, **más precisa y útil será la respuesta**.

In [ ]:
# Prompt vago: sin estructura, resultado impredecible
noticia = "El BCRA subio la tasa de interes de referencia al 40% anual."

prompt_vago = "Analiza esto: " + noticia

print("=== PROMPT ===")
print(prompt_vago)
print()
print("=== RESPUESTA ===")
print(llamar_modelo(prompt_vago))

In [ ]:
# Prompt con los 4 elementos
prompt_completo = (
    "Analiza el siguiente titular de politica monetaria e identifica "
    "sus posibles implicancias para la economia argentina.\n\n"
    "Contexto: La audiencia son estudiantes universitarios de economia que estan "
    "aprendiendo sobre politica monetaria. No asumas conocimientos avanzados.\n\n"
    "Titular a analizar:\n\"" + noticia + "\"\n\n"
    "Formato de salida:\n"
    "- Exactamente 3 puntos breves\n"
    "- Maximo 2 oraciones por punto\n"
    "- Lenguaje claro y accesible"
)

print("=== PROMPT ===")
print(prompt_completo)
print()
print("=== RESPUESTA ===")
print(llamar_modelo(prompt_completo))

> **Observación:** El prompt con estructura guía al modelo hacia exactamente lo que necesitamos.
> El prompt vago puede dar cualquier tipo de respuesta.

---
## 🎛️ 2. Parámetros del Modelo

- **`temperatura`** (0 a 1): controla la variabilidad de la respuesta.
  - **Baja (0.0–0.3)**: respuestas predecibles, consistentes, factuales.
  - **Alta (0.7–1.0)**: respuestas más creativas y variadas.
- **`max_tokens`**: límite de tokens (≈ palabras) en la respuesta.

> **Regla práctica:** temperatura baja para análisis y cálculos; alta para brainstorming.

In [ ]:
tarea = "Sugeri 3 politicas economicas para controlar la inflacion en Argentina."

print("=== TEMPERATURA 0.1 — Respuestas consistentes (correr 2 veces) ===")
print(llamar_modelo(tarea, temperatura=0.1))
print()
print("--- Segunda ejecucion ---")
print(llamar_modelo(tarea, temperatura=0.1))

In [ ]:
print("=== TEMPERATURA 1.0 — Respuestas mas variadas (correr 2 veces) ===")
print(llamar_modelo(tarea, temperatura=1.0))
print()
print("--- Segunda ejecucion ---")
print(llamar_modelo(tarea, temperatura=1.0))

In [ ]:
pregunta = "Cuales son las principales causas de la inflacion en economias emergentes?"

print("=== max_tokens = 60 (respuesta muy corta) ===")
print(llamar_modelo(pregunta, max_tokens=60))
print()
print("=== max_tokens = 400 (respuesta extensa) ===")
print(llamar_modelo(pregunta, max_tokens=400))

---
## 🎭 3. Roles: System, User y Assistant

Los prompts en un LLM se componen de tres **roles**:

- **`system`**: Define el comportamiento, tono y personalidad del modelo antes de cualquier mensaje.
- **`user`**: El mensaje que enviás vos (la persona).
- **`assistant`**: La respuesta generada por el modelo.

El **system prompt** permite transformar al modelo en un asistente especializado
con rol, tono y restricciones específicas.

In [ ]:
pregunta = "Que efectos tiene una suba del tipo de cambio en la economia?"

print("=== SIN SYSTEM PROMPT ===")
print(llamar_modelo(pregunta))

In [ ]:
system_economista = (
    "Sos un economista especializado en macroeconomia y politica monetaria argentina. "
    "Respondés de forma tecnica pero comprensible para estudiantes universitarios. "
    "Siempre usas terminologia economica correcta y alertas sobre incertidumbre cuando corresponde."
)

print("=== CON SYSTEM PROMPT: Economista experto ===")
print(llamar_modelo(pregunta, system=system_economista))

In [ ]:
system_docente = (
    "Sos un docente de economia de escuela secundaria, simpatico y entusiasta. "
    "Explicas los conceptos mas complejos usando analogias del dia a dia que cualquier "
    "adolescente pueda entender. Evitas tecnicismos; si los usas, los explicas de inmediato."
)

print("=== CON SYSTEM PROMPT: Docente de secundaria ===")
print(llamar_modelo(pregunta, system=system_docente))

> **La misma pregunta, tres respuestas completamente distintas.**
> Solo cambiando el system prompt podemos adaptar el modelo a cualquier rol o audiencia.

---
## 🎯 4. Claridad y Especificidad

**Ser claro y directo** es la regla más importante del prompt engineering.

Estrategias clave:
- Usá **verbos de acción**: *"Resumí", "Listá", "Clasificá", "Compará", "Identificá"*
- Especificá el **formato de salida** (cantidad de puntos, longitud, estilo)
- Indicá la **audiencia** objetivo
- Usá **delimitadores** (`---`, comillas, XML tags) para separar instrucciones de datos
- Agregá **guidelines** de calidad (qué debe tener) o **pasos de proceso** (qué debe hacer)

In [ ]:
texto_bcra = (
    "El Banco Central de la Republica Argentina (BCRA) anuncio que mantendra sin cambios "
    "la tasa de politica monetaria en 40% anual. La decision apunta a anclar expectativas "
    "inflacionarias en un contexto de mayor volatilidad cambiaria, con la brecha entre el "
    "dolar oficial y el paralelo superando el 20%."
)

print("=== PROMPT VAGO ===")
print(llamar_modelo("Resumi esto: " + texto_bcra))

In [ ]:
print("=== PROMPT ESPECIFICO con guidelines ===")
prompt_especifico = (
    "Resumi el siguiente comunicado del BCRA para estudiantes universitarios de economia.\n\n"
    + texto_bcra + "\n\n"
    "Instrucciones:\n"
    "- Escribi exactamente 3 puntos clave en formato de lista\n"
    "- Cada punto: maximo 20 palabras\n"
    "- Empieza cada punto con un verbo en infinitivo (Ej: 'Mantener...', 'Indicar...')\n"
    "- Usa terminologia economica correcta pero explica los tecnicismos"
)
print(llamar_modelo(prompt_especifico))

In [ ]:
# Los delimitadores separan claramente instrucciones de datos
titular_1 = "El desempleo bajo al 6,2% en el primer trimestre del anno."
titular_2 = "Las exportaciones de soja cayeron 15% por la sequia."

prompt_delimitadores = (
    "Analiza los siguientes dos titulares economicos. "
    "Para cada uno indica si el impacto en el PBI es Positivo, Negativo o Neutro, y por que.\n\n"
    "---\n"
    "Titular 1: " + titular_1 + "\n"
    "---\n"
    "Titular 2: " + titular_2 + "\n"
    "---\n\n"
    "Formato: Para cada titular, una linea con el veredicto y max 2 oraciones de justificacion."
)
print(llamar_modelo(prompt_delimitadores))

---
## 🏷️ 5. Estructura con Tags XML

Los **tags XML** son delimitadores más explícitos y poderosos.
Son especialmente útiles cuando el prompt incluye:
- Gran cantidad de **datos o contexto**
- **Tipos distintos de contenido** mezclados
- Múltiples **variables interpoladas**

No necesitás conocer XML real — solo usás pares de etiquetas descriptivas:

```
<datos_financieros>  ← mucho mejor que  <data>
<contexto_sectorial> ← mucho mejor que  <info>
<formato_salida>     ← mucho mejor que  <output>
```

Cuanto más descriptivos los nombres, mejor entiende el modelo.

In [ ]:
datos_empresa = (
    "Ventas Q1: $1.200.000. Ventas Q2: $900.000. "
    "Costos Q1: $800.000. Costos Q2: $1.100.000."
)
contexto_empresa = "Empresa del sector alimentario afectada por la sequia de 2023."

print("=== SIN TAGS XML ===")
prompt_sin_xml = (
    "Analiza la situacion financiera de la empresa. "
    "Datos: " + datos_empresa + " "
    "Contexto: " + contexto_empresa + " "
    "Esta en riesgo? Por que?"
)
print(llamar_modelo(prompt_sin_xml))

In [ ]:
print("=== CON TAGS XML ===")
prompt_con_xml = (
    "Analiza la situacion financiera de la empresa y determina su nivel de riesgo.\n\n"
    "<datos_financieros>\n" + datos_empresa + "\n</datos_financieros>\n\n"
    "<contexto_sectorial>\n" + contexto_empresa + "\n</contexto_sectorial>\n\n"
    "<instrucciones>\n"
    "1. Calcula el margen de ganancia de Q1 y Q2 (ganancia = ventas - costos)\n"
    "2. Identifica la tendencia (mejora o deterioro)\n"
    "3. Evalua el nivel de riesgo: Bajo / Medio / Alto, con justificacion concreta\n"
    "</instrucciones>\n\n"
    "<formato_salida>\n"
    "Analisis estructurado en 3 secciones bien marcadas, maximo 150 palabras en total.\n"
    "</formato_salida>"
)
print(llamar_modelo(prompt_con_xml))

---
## 📸 6. Zero-Shot, One-Shot y Few-Shot Prompting

| Técnica | Descripción | Cuándo usarla |
|---|---|---|
| **Zero-Shot** | Solo instrucción, sin ejemplos | Tareas simples y directas |
| **One-Shot** | Instrucción + 1 ejemplo | Para mostrar el formato exacto |
| **Few-Shot** | Instrucción + 3–5 ejemplos | Casos complejos, ironía, formatos específicos |

Los ejemplos muestran al modelo **exactamente** qué output queremos.
Son especialmente útiles para casos difíciles como **ironía o sarcasmo**.

In [ ]:
# Titulares económicos para clasificar: Positivo / Negativo / Neutro
titulares = [
    "El empleo formal crecio un 3% en el primer trimestre del anno.",      # Positivo
    "La tasa de desocupacion trepo al 8,5%, la mas alta en 5 annos.",      # Negativo
    "El BCRA mantuvo sin cambios la tasa de referencia en 40% anual.",     # Neutro
    "Excelente noticia! El peso se devaluo otro 20% esta semana.",         # IRONIA -> Negativo
]

caso_dificil = titulares[3]
print("Titular a clasificar:")
print(" ", caso_dificil)

In [ ]:
print("=== ZERO-SHOT ===")
prompt_zero = (
    "Clasifica el sentimiento del siguiente titular economico.\n"
    "Responde solo con una palabra: Positivo, Negativo o Neutro.\n\n"
    "Titular: \"" + caso_dificil + "\"\n"
    "Sentimiento:"
)
print(llamar_modelo(prompt_zero, temperatura=0.1).strip())

In [ ]:
print("=== ONE-SHOT ===")
prompt_one = (
    "Clasifica el sentimiento del siguiente titular economico.\n"
    "Responde solo con una palabra: Positivo, Negativo o Neutro.\n\n"
    "Ejemplo:\n"
    "Titular: \"El PBI crecio un 4% en el ultimo trimestre.\"\n"
    "Sentimiento: Positivo\n\n"
    "Ahora clasifica:\n"
    "Titular: \"" + caso_dificil + "\"\n"
    "Sentimiento:"
)
print(llamar_modelo(prompt_one, temperatura=0.1).strip())

In [ ]:
print("=== FEW-SHOT (con ejemplo de ironia) ===")
prompt_few = (
    "Clasifica el sentimiento del siguiente titular economico.\n"
    "Responde solo con una palabra: Positivo, Negativo o Neutro.\n"
    "Presta especial atencion a titulares con ironia o sarcasmo.\n\n"
    "Ejemplos:\n"
    "Titular: \"El PBI crecio un 4% en el ultimo trimestre.\"\n"
    "Sentimiento: Positivo\n\n"
    "Titular: \"La tasa de desempleo subio al 9% segun el INDEC.\"\n"
    "Sentimiento: Negativo\n\n"
    "Titular: \"Genial! El precio de los alimentos subio 30% en enero.\"\n"
    "Sentimiento: Negativo\n\n"
    "Ahora clasifica:\n"
    "Titular: \"" + caso_dificil + "\"\n"
    "Sentimiento:"
)
print(llamar_modelo(prompt_few, temperatura=0.1).strip())

In [ ]:
print("=== CLASIFICANDO TODOS LOS TITULARES CON FEW-SHOT ===\n")

ejemplos = (
    "Titular: \"El PBI crecio un 4% en el ultimo trimestre.\"\n"
    "Sentimiento: Positivo\n\n"
    "Titular: \"La tasa de desempleo subio al 9% segun el INDEC.\"\n"
    "Sentimiento: Negativo\n\n"
    "Titular: \"Genial! El precio de los alimentos subio 30% en enero.\"\n"
    "Sentimiento: Negativo"
)

for titular in titulares:
    prompt = (
        "Clasifica el sentimiento del siguiente titular economico.\n"
        "Responde solo con: Positivo, Negativo o Neutro.\n"
        "Presta atencion a la ironia.\n\n"
        "Ejemplos:\n" + ejemplos + "\n\n"
        "Titular: \"" + titular + "\"\n"
        "Sentimiento:"
    )
    resultado = llamar_modelo(prompt, temperatura=0.1).strip()
    print(f"Titular: {titular[:60]}...")
    print(f"  -> {resultado}\n")

---
## 🧠 7. Chain-of-Thought (CoT) — Cadena de Pensamiento

El **Chain-of-Thought prompting** mejora el razonamiento al pedirle al modelo
que "piense en voz alta" antes de dar su respuesta final.

Especialmente efectivo para:
- Problemas matemáticos o financieros
- Razonamiento en múltiples pasos
- Análisis económicos que requieren lógica encadenada

**La técnica más simple — Zero-Shot CoT:**
Solo agregá *"Pensá paso a paso"* al prompt. Sin ejemplos extra. ¡Así de simple!

In [ ]:
problema = (
    "Una empresa tiene ventas nominales de $500.000 y costos nominales de $380.000 "
    "el annio pasado. Si la inflacion anual fue del 60%, "
    "cual es la ganancia real de la empresa (en pesos constantes de hoy)?"
)

print("=== SIN CHAIN-OF-THOUGHT ===")
print(llamar_modelo("Resolve este problema economico: " + problema, temperatura=0.1))

In [ ]:
print("=== CON CHAIN-OF-THOUGHT ===")
prompt_cot = (
    "Resolve el siguiente problema economico.\n\n"
    "Problema:\n" + problema + "\n\n"
    "Pensa paso a paso, mostrando cada calculo intermedio. "
    "Luego da la respuesta final claramente senalada."
)
print(llamar_modelo(prompt_cot, temperatura=0.1, max_tokens=600))

> **Ventajas del CoT:**
> - El razonamiento explícito permite **detectar errores** fácilmente
> - Reduce errores en problemas de **múltiples pasos**
> - Genera respuestas más **auditables y confiables**
>
> Aplicalo a: valor presente, elasticidades, equilibrio de mercado, estados contables.

---
## 📋 8. Prompt Templates con Variables

En la práctica queremos aplicar el **mismo análisis a distintos inputs**.
Los **prompt templates** con f-strings permiten reutilizar prompts garantizando
**consistencia** en los análisis.

Esto es clave para procesar datasets textuales o construir pipelines automatizados.

In [ ]:
def analizar_noticia_economica(titular, cuerpo, audiencia="estudiantes universitarios de economia"):
    """Template reutilizable para analisis de noticias economicas."""
    prompt = (
        "Analiza la siguiente noticia economica argentina.\n\n"
        "<titular>\n" + titular + "\n</titular>\n\n"
        "<cuerpo>\n" + cuerpo + "\n</cuerpo>\n\n"
        "<instrucciones>\n"
        "1. Resume la noticia en 1 oracion (maximo 25 palabras)\n"
        "2. Identifica el indicador economico principal afectado\n"
        "3. Evalua el impacto como: Positivo / Negativo / Neutro\n"
        "4. Justifica en maximo 2 oraciones\n"
        "</instrucciones>\n\n"
        "<audiencia_objetivo>\n" + audiencia + "\n</audiencia_objetivo>"
    )
    return llamar_modelo(prompt, temperatura=0.3)

print("Template definido")

In [ ]:
noticias = [
    {
        "titular": "El INDEC reporto una inflacion mensual de 3,5% en junio",
        "cuerpo": (
            "El IPC registro un incremento del 3,5% en junio, acumulando 42% en el anno. "
            "Alimentos y bebidas lideraron el alza con 4,2% mensual."
        ),
    },
    {
        "titular": "Las exportaciones de bienes aumentaron 12% interanual en mayo",
        "cuerpo": (
            "Las exportaciones alcanzaron USD 7.200 millones en mayo, impulsadas "
            "principalmente por la liquidacion de la cosecha de soja."
        ),
    },
    {
        "titular": "El riesgo pais supero los 1.500 puntos basicos",
        "cuerpo": (
            "El indice EMBI+ Argentina supero hoy los 1.500 puntos, reflejando la "
            "incertidumbre de los inversores ante proximos vencimientos de deuda externa."
        ),
    },
]

for i, noticia in enumerate(noticias, 1):
    print("=" * 60)
    print(f"NOTICIA {i}: {noticia['titular']}")
    print("=" * 60)
    print(analizar_noticia_economica(noticia["titular"], noticia["cuerpo"]))
    print()

---
## 🔄 9. Síntesis: El Prompt Engineering es un Proceso Iterativo

> *"Casi nunca se acierta el prompt a la primera."*

El ciclo es siempre:
1. Escribí un **prompt inicial**
2. Analizá la respuesta: ¿qué falta? ¿qué sobra?
3. **Aplicá una técnica** para mejorarlo
4. Repetí hasta obtener el resultado que necesitás

### Ejercicio integrador

Partimos de un prompt muy básico y lo mejoramos en 3 iteraciones.

In [ ]:
# Iteracion 1: prompt base, muy vago
print("=== ITERACION 1: Prompt base (vago) ===")
print(llamar_modelo("Explicame la elasticidad"))
print()
print("Problema: respuesta muy generica, no sabemos que tipo de elasticidad ni para que contexto.")

In [ ]:
# Iteracion 2: agregamos especificidad + formato
print("=== ITERACION 2: Especificidad + formato de salida ===")
prompt_v2 = (
    "Explica que significa una elasticidad-precio de la demanda de -1,5 para un bien. "
    "Que implicancias tiene para una empresa que quiere aumentar sus precios? "
    "Responde en 3 puntos concisos, maximo 2 oraciones cada uno."
)
print(llamar_modelo(prompt_v2))
print()
print("Mejora: especifico y con formato. Podemos sumar rol + XML + CoT.")

In [ ]:
# Iteracion 3: combinamos rol + XML + CoT + audiencia
print("=== ITERACION 3: Rol + XML + CoT + audiencia definida ===")
prompt_v3 = (
    "Sos un profesor de microeconomia, claro y didactico.\n\n"
    "<contexto>\n"
    "Estamos en clase viendo elasticidades. Los estudiantes entienden oferta y demanda "
    "pero no han visto aplicaciones practicas aun.\n"
    "</contexto>\n\n"
    "<pregunta>\n"
    "Que significa una elasticidad-precio de la demanda de -1,5? "
    "Que deberia hacer una empresa con esa informacion para maximizar sus ingresos?\n"
    "</pregunta>\n\n"
    "<instrucciones>\n"
    "Pensa paso a paso:\n"
    "1. Explica el significado del valor -1,5\n"
    "2. Determina si el bien es elastico o inelastico y que implica eso\n"
    "3. Recomienda la estrategia de precios con justificacion\n"
    "Incluye un ejemplo concreto con numeros para ilustrar cada paso.\n"
    "</instrucciones>"
)
print(llamar_modelo(prompt_v3, temperatura=0.3, max_tokens=700))
print()
print("Tecnicas combinadas: rol + XML + CoT + contexto + audiencia + especificidad")

---
## 📊 Para Seguir Aprendiendo: Evaluación de Prompts

Una vez que dominás las técnicas básicas, el próximo paso es **evaluar prompts sistemáticamente**:
¿Cómo sabemos si un prompt es realmente mejor que otro? ¿Funciona bien en general o solo en un caso?

Para esto existe el campo de **Prompt Evaluation**:
1. Definir **criterios de calidad** para la tarea
2. Generar un **conjunto de casos de prueba** (test set)
3. Ejecutar el prompt en todos los casos
4. **Puntuar las respuestas** (manualmente o con un LLM como juez)

### Recursos recomendados
- Guía de Prompt Engineering de Anthropic: docs.anthropic.com/es/docs/build-with-claude/prompt-engineering/overview
- Prompt Engineering Guide (en español): promptingguide.ai/es
- Google AI Studio: aistudio.google.com — playground para experimentar sin código

---
## 🎓 Resumen

| Técnica | Principio | Cuándo aplicarla |
|---|---|---|
| **4 elementos** | Instrucción + Contexto + Datos + Formato | Siempre — es la base |
| **Temperatura baja** | Respuestas predecibles | Análisis, clasificación, cálculos |
| **Temperatura alta** | Respuestas creativas | Brainstorming, redacción |
| **System prompt / Rol** | El modelo adopta una identidad | Tono o especialización específica |
| **Claridad y especificidad** | Verbos de acción + formato + audiencia | Siempre |
| **Tags XML** | Separar tipos de contenido | Prompts complejos con múltiples datos |
| **Few-Shot** | Ejemplos que muestran el output | Casos difíciles, ironía, formatos específicos |
| **Chain-of-Thought** | "Pensá paso a paso" | Problemas con múltiples pasos |
| **Templates** | F-strings con variables | Análisis consistente de múltiples inputs |

---
> **Clave:** El prompt engineering es **iterativo**. Empezá simple → analizá → aplicá una técnica → repetí.